This notebook leverages the ESM Metagenomic Atlas API to predict 3D protein structures from primary amino acid sequences. It is optimized for high-throughput batch processing of FASTA files, providing automated sequence validation, error handling, and result aggregation.
**Feature,Limit / Requirement**



*   Max Sequence Length,400 amino acids
*   Input Format,"Standard Multi-FASTA (.fasta, .fa)"
*   API Rate Limit, 15s delay between requests
*   Output Format,Protein Data Bank (.pdb) files

If you use this please cite,

Lin, Z., Akin, H., Rao, R., Hie, B., Zhu, Z., Lu, W., Smetanin, N., Verkuil, R., Kabeli, O., Shmueli, Y., dos Santos Costa, A., Fazel-Zarandi, M., Sercu, T., Candido, S., & Rives, A. (2023). Evolutionary-scale prediction of atomic-level protein structure with a language model. Science, 379(6637), 1123–1130. https://doi.org/10.1126/science.ade2574








In [ ]:
#Installating dependencies
!pip install -q biopython requests ipywidgets

In [ ]:
import io
import time
import requests
import shutil
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from Bio import SeqIO

FOLDER_NAME = "folded_proteins"
API_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"
SECONDS_BETWEEN = 15.0

def start_folding_process(file_content):
    if os.path.exists(FOLDER_NAME):
        shutil.rmtree(FOLDER_NAME)
    os.makedirs(FOLDER_NAME)

    fasta_text = file_content.decode('utf-8')
    proteins = list(SeqIO.parse(io.StringIO(fasta_text), "fasta"))
    total_count = len(proteins)

    pbar = widgets.IntProgress(
        value=0,
        min=0,
        max=total_count,
        description='Progress:',
        bar_style='info'
    )
    display(pbar)

    for i, protein in enumerate(proteins):
        clean_id = "".join(c for c in protein.id if c.isalnum() or c in "._-").strip()
        sequence = str(protein.seq).replace("*", "").upper().strip()

        if len(sequence) > 400:
            print(f"Skipping {clean_id}: Length {len(sequence)} exceeds 400aa limit.")
            pbar.value = i + 1
            continue

        print(f"[{i+1}/{total_count}] Processing: {clean_id}")

        try:
            response = requests.post(API_URL, data=sequence, timeout=120)

            if response.status_code == 200:
                filepath = os.path.join(FOLDER_NAME, f"{clean_id}.pdb")
                with open(filepath, "w") as f:
                    f.write(response.text)
            else:
                print(f"API Error ({response.status_code}) for {clean_id}: {response.reason}")

        except Exception as e:
            print(f"Connection error for {clean_id}: {str(e)}")

        pbar.value = i + 1
        if i < total_count - 1:
            time.sleep(SECONDS_BETWEEN)

    if os.listdir(FOLDER_NAME):
        shutil.make_archive(FOLDER_NAME, 'zip', FOLDER_NAME)
        print(f"\nExecution complete. Archive created: {FOLDER_NAME}.zip")
    else:
        print("\nExecution complete. No PDB files were generated.")

uploader_btn = widgets.FileUpload(
    accept='.fasta, .fa',
    multiple=False,
    description="Select FASTA",
    button_style='primary'
)
output_view = widgets.Output()

def on_upload(change):
    with output_view:
        clear_output()
        if not uploader_btn.value:
            print("No file detected.")
            return
        files = uploader_btn.value
        if isinstance(files, dict):
            for filename, file_info in files.items():
                print(f"Input file: {filename}")
                start_folding_process(file_info['content'])
        else:
            for file_info in files:
                print(f"Input file: {file_info['name']}")
                start_folding_process(file_info['content'])

uploader_btn.observe(on_upload, names='value')

print("ESMFold Batch Submission System")
display(uploader_btn, output_view)

ESMFold Batch Submission System


FileUpload(value={}, accept='.fasta, .fa', button_style='primary', description='Select FASTA')

Output()